# XRF map-averaged spectrum — constrained Voigt fit and quantification

Reads a BLISS mesh scan, corrects the MCA spectra for incident flux and detector
dead time, forms the map-averaged spectrum, fits it with a **constrained** Voigt
model, quantifies it with a fundamental-parameter matrix correction, and maps
the result pixel by pixel.

## The model

One free amplitude per *line group*, where a group is one (element, shell family)
— e.g. `Nd L`, `Cu K`. That amplitude is the integrated area of the group's most
intense line; every other line of the group is tied to it through its computed
relative intensity. Line positions are never free: they come from the tabulated
energies and the fixed calibration. Widths come from a two-parameter detector
resolution law shared by all lines. Each line carries its **Si escape peaks**
(no extra parameter), and a **pile-up** term built from the per-pixel
self-convolution of the measured spectra is added with one resolving time.

## Relative intensities and sensitivities

Computed, not tabulated, from four factors:

1. `xraylib.CS_FluorLine_Kissel` partial fluorescence cross-sections (subshell
   ionisation, Coster-Kronig, fluorescence yields) — **not** `RadRate`, which is
   normalised within a subshell and cannot be used across an L series;
2. the BM32 white beam (`xrf_beamline.py`), integrated as `int sigma(E) dPhi/dE dE`
   with the per-0.1 %BW -> per-keV conversion;
3. the detection efficiency of the KETEK VITUS H7 CH (sample-detector air path,
   1 µm C window, 450 µm Si);
4. the sample matrix (`xrf_matrix.py`): primary absorption along the incident
   and emitted paths, and secondary fluorescence, for a stack of layers
   (e.g. cap / film / substrate), each of finite thickness or bulk.

## Out of scope for now

* **Energy calibration** is taken as fixed (`E[eV] = 10.032 x channel - 31.263`).
* **Detector tail/shelf** (Hypermet): the free Lorentzian width stands in for it.
* **Laue (Bragg) peaks** of the sample in the fluorescence spectrum.

## 1. Configuration

In [ ]:
# %matplotlib widget      # interactive pan/zoom; needs ipympl
%matplotlib inline

import json
import logging
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import xraylib
from scipy.integrate import trapezoid

from bliss_hdf5_reader import BlissMeshScan
from xrf_beamline import BM32_WHITE_BEAM, per_unit_energy
from xrf_fit import (Calibration, Detector, element_line_groups, normalise_spectra, fit_map,
                     plot_pixel_fits, select_pixels,
                     average_spectrum, counting_error, fit_average_spectrum,
                     plot_fit, identify_peaks, number_ratio,
                     pileup_spectra, average_pileup_template,
                     width_window, peak_width_profile, detect_steps, segment_labels,
                     fit_segments)
from xrf_matrix import Geometry, Layer, Stack, MatrixModel, quantify_layer, atom_ratio

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")

# ---------------------------------------------------------------- input data
# At the beamline, paths look like
#   /data/visitor/<proposal>/bm32/<date>/RAW_DATA/<sample>/<sample>_<scan>/<...>.h5
#
# Per sample: the elements to fit; the sample as a layer stack, SURFACE FIRST
# (Layer(composition, density g/cm3, thickness um or None for bulk, name)), with
# `target` the index of the layer being quantified -- its composition is only
# the STARTING guess and is iterated, the other layers are held fixed and their
# emission/absorption subtracted; the reference group against which atom ratios
# are expressed; unmeasured elements fixed per reference atom (e.g. O).
SAMPLES = {
    "Nd2Ir2O7_bulk": dict(
        h5=Path("NIO295_laue_map_precise.h5"), scan=1,
        elements=["Nd", "Ir", "Pt", "Pb"],     # Pb is beamline contamination
        roi_ev=(4300.0, 15800.0), bkg_order=2,
        stack=[Layer("Nd2Ir2O7", 9.35, None, "crystal")],   # pyrochlore, a ~ 10.37 A
        target=0,
        reference="Ir L", analytes=["Nd L"], fixed_atoms={"O": 3.5},
        closure=None,
        note="bulk single crystal — self-absorption and secondary fluorescence matter",
    ),
    "InGaAs_film": dict(
        h5=Path("InPoSi_v2_center.h5"), scan=1,
        elements=["In", "Ga", "As", "Cu", "Ni", "Ca", "Ar"],   # Cu, Ni, Ca: contamination; Ar: air (chi2 1654 -> 1529 with Ca, Ar)
        roi_ev=(2900.0, 12600.0), bkg_order=3,
        stack=[Layer("InP", 4.81, 0.05, "cap"),             # 50 nm InP cap
               Layer("In0.5Ga0.5As", 5.5, 1.9, "film"),    # thickness estimate
               Layer("Si", 2.33, None, "substrate")],
        target=1,
        reference="As K", analytes=["Ga K", "In L"], fixed_atoms={},
        closure=["Ga", "In"],      # III-V: n_Ga + n_In = n_As
        note="50 nm InP / 1.9 um (In,Ga)As / Si",
    ),
}
SAMPLE = "InGaAs_film"          # <- pick the sample here

_s = SAMPLES[SAMPLE]
H5_FILE     = _s["h5"]
SCAN_NUMBER = _s["scan"]
MCA         = "xia_det0"
MONITOR     = "mon"
LIVE_TIME   = f"{MCA}_live_time"
REAL_TIME   = f"{MCA}_elapsed_time"      # for the pile-up rate
print(f"sample: {SAMPLE} — {_s['note']}")

OUT_DIR = Path("processed")                        # at the beamline: .../PROCESSED_DATA
OUT_DIR.mkdir(exist_ok=True)

# ------------------------------------------------------------ physics setup
CALIB = Calibration(gain=10.032, offset=-31.263)   # E[eV] = gain*channel + offset

# Ambient air (incident path and detection path)
AIR_PATH_M    = 3.3        # incident air path, measured on BM32
AIR_TEMP_C    = 20.0
AIR_PRESS_HPA = 1013.25    # local pressure; ~990 hPa at ESRF altitude

# KETEK VITUS H7 CH: 450 um Si, 1 um carbon window; ~20 cm sample-detector air.
# Resolution: 135 eV at Mn Ka sets the electronic-noise term at the stated Fano factor.
DETECTOR = Detector.from_fwhm(fwhm_ev=135.0, at_energy_ev=5900.0,
                              material="Si", fano=0.12, gamma_ev=5.0,
                              sensor_thickness_um=450.0,
                              window_material="C", window_thickness_um=1.0,
                              air_path_cm=20.0, air_temperature_c=AIR_TEMP_C,
                              air_pressure_hpa=AIR_PRESS_HPA)

# Measurement geometry, angles from the sample SURFACE.  TO CONFIRM: the In L
# result depends strongly on the take-off angle (see the closure test).
GEOMETRY = Geometry(incidence_deg=40.0, takeoff_deg=40.0)

ELEMENTS   = _s["elements"]
FAMILIES   = ("K", "L")
# Physics switches.  Each one changes relative intensities within groups and
# the sensitivities used for quantification.
USE_WHITE_BEAM          = True     # int sigma dPhi/dE dE; False -> single energy E0
USE_DETECTOR_EFFICIENCY = True
USE_MATRIX              = True     # self-absorption in the layer / bulk
SECONDARY_FLUORESCENCE  = True     # only used with USE_MATRIX
USE_ESCAPE              = True     # Si escape peaks, tied to their parent lines
USE_PILEUP              = True     # sum peaks from per-pixel self-convolution
PILEUP_ROI_EV           = (2900.0, 20300.0)   # range used to fit the resolving time
N_REFIT                 = 1        # refits at the converged composition
# Detector-resolution changes during the scan (see section 10b):
#   "none"     one set of line widths for the whole map
#   "segments" widths fitted per segment between detected steps (abrupt changes)
#   "linear"   one first-order width term per pixel (small steps and drifts)
#   "both"     segments, plus the linear term for what is left
RESOLUTION_MODE         = "both"
STEP_THRESHOLD          = 6.0      # step detection, in units of the row-to-row noise
STEP_MIN_EV             = 1.0      # ignore width steps smaller than this (rms width, eV)

E0_EFF_KEV = 20.0        # only used when USE_WHITE_BEAM is False
E_GRID_KEV = np.arange(3.0, 40.0 + 1e-9, 0.010)   # 10 eV grid for the beam model
MIN_REL    = 0.005       # drop lines below this fraction of their group reference
ROI_EV     = _s["roi_ev"]
BKG_ORDER  = _s["bkg_order"]   # polynomial background order over the ROI

print(f"detector: noise = {DETECTOR.noise_ev:.1f} eV, Fano = {DETECTOR.fano}, "
      f"gamma = {DETECTOR.gamma_ev} eV")
for e in (5900, 9170, 12500):
    print(f"  FWHM({e/1000:5.2f} keV) = {DETECTOR.fwhm_ev(e):6.1f} eV")
print("detection efficiency:", "  ".join(f"{e/1000:.2f} keV {DETECTOR.efficiency(e):.3f}"
                                       for e in (3300, 6000, 9250, 10540, 12500)))

## 2. Load the scan

In [ ]:
scan = BlissMeshScan(h5_filepath=H5_FILE, scan_number=SCAN_NUMBER, verbose=False)
scan.read_data(extra_counters=[MCA, MONITOR, LIVE_TIME, REAL_TIME, f"{MCA}_fractional_dead_time"])

print(scan.get_summary())

spectra = scan.counters[MCA].astype(float)        # (n_pixels, n_channels)
n_pixels, n_channels = spectra.shape
print(f"\nMCA: {n_pixels} pixels x {n_channels} channels, {spectra.sum():.3e} counts total")

# Two warnings above are expected and harmless on a fluorescence-only analysis:
#   - the eiger image folder is absent (no Laue images needed here);
#   - the 2D MCA array is skipped by the 1D grid reshaper.

## 3. Normalisation: incident flux and detector dead time

Both corrections are per pixel and multiplicative:
`y_corrected = y_raw / (monitor_rel * live_time)`.

Dividing by the XIA live time converts recorded counts into a true incoming
rate. On white-beam maps this is the larger of the two corrections by far, and
it is spatially structured — it tracks total scattering power, so it does **not**
cancel in an elemental ratio.

In [ ]:
monitor   = scan.counters[MONITOR]
live_time = scan.counters[LIVE_TIME]
fdt       = scan.counters[f"{MCA}_fractional_dead_time"]

corrected, scale = normalise_spectra(spectra, monitor=monitor, live_time=live_time)

print(f"monitor        : {monitor.min():.4g} - {monitor.max():.4g} "
      f"(spread {100*np.ptp(monitor)/monitor.mean():.1f} %)")
print(f"live time (s)  : {live_time.min():.4g} - {live_time.max():.4g}")
print(f"dead-time frac.: {fdt.min():.3f} - {fdt.max():.3f}, median {np.median(fdt):.3f}")
print(f"-> correction factor spans {1/(1-fdt.max()):.2f}x across the map")

fig, axes = plt.subplots(1, 3, figsize=(14, 3.6))
for ax, (data, title) in zip(axes, [(monitor, "monitor"),
                                    (live_time, "XIA live time (s)"),
                                    (fdt, "fractional dead time")]):
    im = ax.imshow(data.reshape(scan.scan_shape), cmap="viridis", origin="lower")
    ax.set_title(title)
    ax.set_xlabel(f"{scan.fast_motor_name} (pixel)")
    ax.set_ylabel(f"{scan.slow_motor_name} (pixel)")
    fig.colorbar(im, ax=ax)
fig.suptitle("Per-pixel normalisation terms", y=1.02)
fig.tight_layout()
fig.savefig(OUT_DIR / f"{H5_FILE.stem}_normalisation.png", dpi=150, bbox_inches="tight")

## 4. Map-averaged spectrum

Two uncertainty estimates are computed, and they answer different questions:

* **counting error** — the Poisson error of the average, propagated through the
  normalisation. This is the right weight for *fitting* the average spectrum.
* **standard error of the mean** — the pixel-to-pixel spread divided by sqrt(N).
  Much larger than the counting error wherever the map is genuinely
  heterogeneous, so it is a map-structure diagnostic, not a fit weight.

In [ ]:
avg, sem, n_used = average_spectrum(corrected)
cnt_err = counting_error(spectra, scale)

lo, hi = CALIB.channel_range(*ROI_EV, n_channels)
ROI = (lo, hi)
print(f"pixels used: {n_used}/{n_pixels}")
print(f"ROI: channels {lo}-{hi} = {CALIB.energy(lo)/1000:.2f}-{CALIB.energy(hi-1)/1000:.2f} keV")
print(f"median counting error / peak amplitude = {np.median(cnt_err[lo:hi])/avg[lo:hi].max():.2e}")
print(f"median SEM            / peak amplitude = {np.median(sem[lo:hi])/avg[lo:hi].max():.2e}")

ch = np.arange(n_channels)
fig, ax = plt.subplots(figsize=(11, 4.5))
ax.fill_between(CALIB.energy(ch)/1000, avg - sem, avg + sem, color="C0", alpha=0.3,
                lw=0, label=r"pixel-to-pixel spread ($\pm$ SEM)")
ax.plot(CALIB.energy(ch)/1000, avg, "k-", lw=0.9, label="map-averaged spectrum")
ax.axvspan(CALIB.energy(lo)/1000, CALIB.energy(hi-1)/1000, color="0.9", zorder=0, label="fit ROI")
ax.set_yscale("log")
ax.set_xlabel("Energy (keV)")
ax.set_ylabel("Counts per live second (monitor-normalised)")
ax.set_xlim(0, CALIB.energy(n_channels-1)/1000)
ax.set_ylim(max(avg[avg > 0].min(), 1e-3), avg.max()*3)
ax.legend()
ax.set_title(f"{H5_FILE.stem} — scan {SCAN_NUMBER}")
fig.tight_layout()
fig.savefig(OUT_DIR / f"{H5_FILE.stem}_average_spectrum.png", dpi=150, bbox_inches="tight")

## 5. Which elements? — peak scouting

`identify_peaks` lists tabulated lines whose energy is close to each detected
peak, ranked by radiative rate. It is a *scouting* tool: a close energy match is
suggestive, never proof. Confirmation comes from the fit — a genuine element
accounts for its whole line series at the tabulated ratios, a coincidence does
not.

In [ ]:
for p in identify_peaks(avg, CALIB, ROI, prominence_frac=0.01):
    print(f"ch {p['channel']:4d}  {p['energy_ev']/1000:6.3f} keV  h={p['height']:8.1f}")
    for c in p["candidates"]:
        print(f"        {c}")

## 6. White-beam excitation, detection and matrix

The beamline is modelled as a bending-magnet source propagated through its
optics (`xrf_beamline.py`), giving the incident spectrum at the sample. Its flux
is per 0.1 % bandwidth, i.e. per `dE = 1e-3 E`, so every integral over energy
uses the per-keV density `dPhi/dE = Phi_0.1%BW / (1e-3 E)`.

Line strengths then become flux-weighted integrals `int sigma_l(E) dPhi/dE dE`
instead of cross-sections at one energy. Each L subshell is excited only by the
part of the spectrum above *its own* edge, and for the heavy elements here those
edges sit inside the band (Ir L3/L2/L1 at 11.22/12.82/13.42 keV; Pb at
13.04/15.20/15.86). The mirrors are **Ir coated**, so the beam carries the Ir L
edges of the optics at exactly the energies where the sample's Ir L shells are
excited — visible as steps in the spectrum.

On top of the excitation come the detection efficiency (which favours lines
travelling through less air) and the matrix: absorption of the incident and
emitted radiation in the sample, and secondary fluorescence (e.g. As K -> Ga K,
Ga/As K -> In L). The matrix term couples to the incident spectrum inside the
energy integral, so the three cannot be applied as separate factors.

In [ ]:
beamline = BM32_WHITE_BEAM(air_path_m=AIR_PATH_M, temperature_c=AIR_TEMP_C,
                           pressure_hpa=AIR_PRESS_HPA)
raw_flux, phi, transfers = beamline.spectrum(E_GRID_KEV)      # per 0.1 % BW
dens = per_unit_energy(E_GRID_KEV, phi)                        # per keV
e_lo, e_hi = beamline.band_limits(E_GRID_KEV, frac=0.10)

print(f"source field         : {beamline.source.magnetic_field_t} T")
print(f"critical energy      : {beamline.source.critical_energy_kev:.2f} keV")
print(f"band at 10 % of peak : {e_lo:.2f} - {e_hi:.2f} keV")
print(f"band at  1 % of peak : {beamline.band_limits(E_GRID_KEV, 0.01)[0]:.2f} - "
      f"{beamline.band_limits(E_GRID_KEV, 0.01)[1]:.2f} keV")
print(f"flux (per 0.1% BW) peaks at {E_GRID_KEV[np.argmax(phi)]:.2f} keV; "
      f"photons per keV peak at {E_GRID_KEV[np.argmax(dens)]:.2f} keV")

# The low-energy edge is a sensitive measure of the air path, so a quoted band
# edge can be inverted for the path length. Useful as a consistency check.
from scipy.optimize import brentq
E_COARSE = np.linspace(3.0, 40.0, 500)
for target in (5.0, 6.0, 7.0):
    try:
        L = brentq(lambda x: BM32_WHITE_BEAM(air_path_m=x, temperature_c=AIR_TEMP_C,
                                             pressure_hpa=AIR_PRESS_HPA
                                             ).band_limits(E_COARSE, 0.01)[0] - target,
                   0.02, 8.0, xtol=1e-3)
        print(f"  a 1 % band edge at {target:.1f} keV would imply {L:.2f} m of air")
    except ValueError:
        pass

total = trapezoid(dens, x=E_GRID_KEV)
print("\nfraction of incident PHOTONS above each L edge:")
for el in ELEMENTS:
    z = xraylib.SymbolToAtomicNumber(el)
    parts = []
    for shell, nm in [(xraylib.L3_SHELL, "L3"), (xraylib.L2_SHELL, "L2"), (xraylib.L1_SHELL, "L1")]:
        edge = xraylib.EdgeEnergy(z, shell)
        m = E_GRID_KEV >= edge
        parts.append(f"{nm} ({edge:5.2f} keV) {100*trapezoid(dens[m], x=E_GRID_KEV[m])/total:5.1f} %")
    print(f"  {el}: " + "   ".join(parts))

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 7), sharex=True,
                               gridspec_kw={"height_ratios": [2, 1], "hspace": 0.1})
ax1.plot(E_GRID_KEV, raw_flux/raw_flux.max(), "k--", lw=1.3, label="bending-magnet source")
ax1.plot(E_GRID_KEV, phi/raw_flux.max(), "r-", lw=1.8, label="at sample, after optics")
ax1.axvspan(e_lo, e_hi, color="C1", alpha=0.12, zorder=0, label=f"10 % band: {e_lo:.1f}-{e_hi:.1f} keV")
ax1.axvline(beamline.source.critical_energy_kev, color="C0", ls=":", lw=1.2,
            label=f"$E_c$ = {beamline.source.critical_energy_kev:.1f} keV")
ax1.set_ylabel("Flux per 0.1 % BW\n(normalised to source peak)")
ax1.set_title("Incident white-beam spectrum")
ax1.legend(fontsize=9)
for lab, t in transfers:
    ax2.plot(E_GRID_KEV, t, lw=1.1, label=lab)
ax2.set_xlabel("Energy (keV)"); ax2.set_ylabel("Reflectivity /\ntransmission")
ax2.set_yscale("log"); ax2.set_ylim(1e-8, 2.0)   # the air path spans ~7 decades
ax2.legend(fontsize=7, ncol=2)
fig.savefig(OUT_DIR / "BM32_white_beam_spectrum.png", dpi=150, bbox_inches="tight")

In [ ]:
# Line groups under increasing levels of physics.  Each level changes relative
# intensities inside groups (never positions):
#   scalar E0 -> white beam -> + detection efficiency -> + matrix (absorption, SF)
STACK = Stack(_s["stack"])
matrix0 = MatrixModel(STACK, GEOMETRY, E_GRID_KEV, phi, secondary=SECONDARY_FLUORESCENCE,
                      target=_s["target"])
print(STACK, "| target:", STACK[_s["target"]].name or _s["target"], "|", GEOMETRY)

if not USE_WHITE_BEAM:
    LEVEL = "scalar"
elif not USE_MATRIX:
    LEVEL = "white+eff" if USE_DETECTOR_EFFICIENCY else "white"
else:
    LEVEL = "matrix"

def build_groups(elements=ELEMENTS, level=None, matrix=None):
    """Line groups for the chosen physics level (default: the configured one)."""
    level = level or LEVEL
    kw = dict(families=FAMILIES, min_rel=MIN_REL)
    det = DETECTOR if USE_DETECTOR_EFFICIENCY else None
    if level == "scalar":
        return element_line_groups(elements, *ROI_EV, e0_eff_kev=E0_EFF_KEV, **kw)
    if level == "white":
        return element_line_groups(elements, *ROI_EV, incident_spectrum=(E_GRID_KEV, phi), **kw)
    if level == "white+eff":
        return element_line_groups(elements, *ROI_EV, incident_spectrum=(E_GRID_KEV, phi),
                                   detector=DETECTOR, **kw)
    return element_line_groups(elements, *ROI_EV, detector=det, matrix=matrix or matrix0, **kw)

t0 = time.time()
VARIANTS = {lvl: build_groups(level=lvl) for lvl in ("scalar", "white", "white+eff", "matrix")}
print(f"line groups built for 4 physics levels in {time.time()-t0:.1f} s\n")

print(f"{'group':<7}{'line':<7}{'E (eV)':>8}" + "".join(f"{k:>11}" for k in VARIANTS))
for i, g in enumerate(VARIANTS["scalar"]):
    rels = [{l.name: l.rel for l in VARIANTS[k][i].lines} for k in VARIANTS]
    for l in g.lines:
        if max(r.get(l.name, 0) for r in rels) > 0.05:
            print(f"{g.label:<7}{l.name:<7}{l.energy_ev:>8.0f}"
                  + "".join(f"{r.get(l.name, float('nan')):>11.3f}" for r in rels))
print("\nThe reference line has rel = 1 by construction; the others move with the physics.")

### Pile-up and escape peaks

**Escape.** A photon absorbed in the Si sensor can be followed by a Si K X-ray
that leaves the crystal, so a fraction of each line (~0.1 % for Ga/As K, ~1 %
for In L) is recorded 1.74 keV lower. These escape peaks are tied to their
parent lines; the group amplitude counts parent + escape.

**Pile-up.** Two photons within the pulse-pair resolving time `tau` are recorded
as one at the summed energy. For each pixel the expected pile-up spectrum is the
self-convolution of its measured spectrum divided by the real time; the
map-averaged template carries one free parameter, `tau`. The main sum peaks
(2 x Ga Ka at 18.5 keV, Ga Ka + As Ka at 19.8 keV, ...) usually lie above the
analysis ROI, so `tau` is fitted once over `PILEUP_ROI_EV` and then fixed.
Pile-up removes events from every line with the same probability, so it does
not bias line ratios — it only adds sum peaks that would otherwise be absorbed
by other components.

In [ ]:
real_time = scan.counters[REAL_TIME]
PILEUP = pileup_spectra(spectra, CALIB, real_time)            # per pixel, counts / s
PILEUP_TMPL = average_pileup_template(PILEUP, scale)          # per µs, data units

TAU_US = 0.0
if USE_PILEUP:
    roi_pp = CALIB.channel_range(*PILEUP_ROI_EV, n_channels)
    g_pp = element_line_groups(ELEMENTS, *PILEUP_ROI_EV, families=FAMILIES, min_rel=MIN_REL,
                               incident_spectrum=(E_GRID_KEV, phi), detector=DETECTOR)
    for bo in (BKG_ORDER, BKG_ORDER + 2):
        r_pp = fit_average_spectrum(avg, g_pp, CALIB, DETECTOR, roi_pp, yerr=cnt_err,
                                    background_order=bo, escape=USE_ESCAPE,
                                    pileup_template=PILEUP_TMPL)
        print(f"pile-up fit over {PILEUP_ROI_EV[0]/1e3:.1f}-{PILEUP_ROI_EV[1]/1e3:.1f} keV, "
              f"background order {bo}: tau = {r_pp.model.pileup_tau_us(r_pp.popt):.4f} us, "
              f"chi2_red = {r_pp.chi2_red:.1f}")
    TAU_US = float(r_pp.model.pileup_tau_us(r_pp.popt))     # the higher-order background
    _hi = CALIB.energy(np.arange(n_channels)) > 2000.0        # skip the sub-keV noise peak
    e_top = CALIB.energy(np.flatnonzero(_hi)[np.argmax(PILEUP_TMPL[_hi])]) / 1000
    print(f"-> tau fixed at {TAU_US:.4f} us; strongest sum feature at {e_top:.2f} keV")

# keyword arguments shared by every average-spectrum fit below
FIT_KW = dict(yerr=cnt_err, background_order=BKG_ORDER, escape=USE_ESCAPE,
              pileup_template=PILEUP_TMPL if USE_PILEUP else None,
              fit_pileup=False, pileup_tau_us=TAU_US)

In [ ]:
# ...and what does each level do to the fit?  (escape and pile-up as configured)
print(f"{'level':<11}{'chi2_red':>10}{'resid rms %':>13}   reference-line areas")
for lvl, grp in VARIANTS.items():
    r = fit_average_spectrum(avg, grp, CALIB, DETECTOR, ROI, **FIT_KW)
    rr = np.abs(r.residual) / r.y.max()
    print(f"{lvl:<11}{r.chi2_red:>10.1f}{100*np.sqrt((rr**2).mean()):>13.3f}   "
          + "  ".join(f"{k}={v:.4g}" for k, v in r.areas.items()))

# Matrix effects at the starting composition, for each group's reference line
if USE_MATRIX:
    print(f"\ntarget layer {STACK[_s['target']]}, {GEOMETRY}:")
    print(f"  {'group':<7}{'line':<7}{'E (keV)':>8}{'absorption':>12}{'SF / primary':>14}   main exciters")
    for g in VARIANTS["matrix"]:
        z = xraylib.SymbolToAtomicNumber(g.element)
        macro = getattr(xraylib, g.reference.name + "_LINE")
        e = g.reference.energy_ev / 1000.0
        p = matrix0.primary(z, macro, e)
        sec, parts = matrix0.secondary(z, macro, e, by_exciter=True)
        top = sorted(parts.items(), key=lambda kv: -kv[1])[:3]
        print(f"  {g.label:<7}{g.reference.name:<7}{e:>8.3f}{matrix0.absorption_factor(z, macro, e):>12.3f}"
              f"{100*sec/p:>13.2f}%   " + ", ".join(f"{k} {100*v/p:.2f}%" for k, v in top if v > 0))
    print("  absorption = primary intensity / absorption-free layer of the same mass "
          "thickness, overlayers included (NaN for bulk)")
    print("  exciters are labelled <layer index>:<element> <line>")

## 7. Build the line groups

Each group's lines are listed relative to its most intense line (`rel = 1`),
which is the one carrying the free amplitude.

In [ ]:
groups = build_groups()
print("physics level:", LEVEL, "| escape:", USE_ESCAPE, "| pile-up:", USE_PILEUP,
      f"(tau = {TAU_US:.3f} us, fixed)" if USE_PILEUP else "")

for g in groups:
    print(f"\n{g.label}  ({len(g.lines)} lines, sum of relative intensities = {g.total_rel:.3f})")
    print(f"  {'line':<8}{'E (eV)':>10}{'rel':>9}")
    for line in g.lines:
        print(f"  {line.name:<8}{line.energy_ev:>10.0f}{line.rel:>9.3f}")

n_free = len(groups) + (BKG_ORDER + 1) + 3
print(f"\n{sum(len(g.lines) for g in groups)} lines modelled with {n_free} free parameters")

## 8. Fit

In [ ]:
result = fit_average_spectrum(avg, groups, CALIB, DETECTOR, ROI, **FIT_KW)

# Self-consistency: absorption and secondary fluorescence depend on the
# composition being measured.  Iterate the composition on the fitted areas,
# rebuild the line groups (their relative intensities depend on the matrix),
# refit, and quantify again.
QUANT = None
if USE_MATRIX:
    QUANT = quantify_layer(result.areas, {g.label: g for g in groups}, matrix0, DETECTOR,
                           reference=_s["reference"], analytes=_s["analytes"],
                           fixed_atoms=_s["fixed_atoms"])
    for _ in range(N_REFIT):
        groups = build_groups(matrix=QUANT["matrix"])
        result = fit_average_spectrum(avg, groups, CALIB, DETECTOR, ROI, **FIT_KW)
        QUANT = quantify_layer(result.areas, {g.label: g for g in groups}, QUANT["matrix"],
                               DETECTOR, reference=_s["reference"], analytes=_s["analytes"],
                               fixed_atoms=_s["fixed_atoms"])
    print(f"converged layer: {QUANT['layer']}  ({QUANT['iterations']} iterations)\n")

print(result.summary())

resid_rel = np.abs(result.residual) / result.y.max()
resid_loc = np.abs(result.residual) / np.maximum(result.y, 1e-12)
print(f"\nresidual vs PEAK amplitude : rms {100*np.sqrt((resid_rel**2).mean()):.2f} %, "
      f"max {100*resid_rel.max():.2f} %")
print(f"residual vs LOCAL data     : median {100*np.median(resid_loc):.1f} %")
# The second number is the honest one for weak lines. A spectrum with one
# dominant peak can show a tiny rms/peak while the model is badly wrong in the
# low-signal regions where a trace element sits.

A note on reduced chi-squared. Averaging thousands of pixels makes the counting
error on the mean spectrum ~0.01 % of the peak amplitude, so any model
imperfection at the per-cent level gives chi2_red in the thousands. Here it
measures **model inadequacy, not noise**, and the parameter errors it reports are
correspondingly optimistic. Judge the fit on the residual as a fraction of peak
amplitude, and on whether the residual is structured.

In [ ]:
fig, (ax, axr) = plot_fit(result, energy_axis=True, logy=True)
fig.savefig(OUT_DIR / f"{H5_FILE.stem}_voigt_fit.png", dpi=150, bbox_inches="tight")

## 9. How much does the element list matter?

Overlapping series compete for the same counts, so dropping one element biases
the others. This leave-one-out test quantifies that — it is the honest way to
report how well-determined each area is, and it matters more than the formal fit
errors.

In [ ]:
ref = result.areas
rows = []
for drop in ELEMENTS:
    kept = [e for e in ELEMENTS if e != drop]
    g2 = build_groups(kept, matrix=QUANT["matrix"] if QUANT else None)
    r2 = fit_average_spectrum(avg, g2, CALIB, DETECTOR, ROI, **FIT_KW)
    rr = np.abs(r2.residual) / r2.y.max()
    shifts = ", ".join(f"{k} {100*(r2.areas[k]-ref[k])/ref[k]:+.1f} %" for k in r2.areas)
    rows.append((drop, r2.chi2_red, 100*np.sqrt((rr**2).mean()), shifts))

print(f"{'dropped':<10}{'chi2_red':>12}{'resid rms %':>14}   shift in remaining areas")
print(f"{'(none)':<10}{result.chi2_red:>12.1f}"
      f"{100*np.sqrt((resid_rel**2).mean()):>14.2f}   -")
for drop, c2, rms, shifts in rows:
    print(f"{drop:<10}{c2:>12.1f}{rms:>14.2f}   {shifts}")

## 10. Save

In [ ]:
stem = H5_FILE.stem
np.savez_compressed(
    OUT_DIR / f"{stem}_average_spectrum.npz",
    channel=np.arange(n_channels), energy_ev=CALIB.energy(np.arange(n_channels)),
    average=avg, counting_error=cnt_err, sem=sem,
    roi=np.array(ROI), fit_channels=result.channels, fit_model=result.yfit,
    background=result.model.background(result.popt),
    pileup=result.model.pileup_profile(result.popt),
    **{f"component_{g.label.replace(' ', '')}": result.model.group_profile(i, result.popt)
       for i, g in enumerate(result.model.groups)},
)

record = {
    "file": str(H5_FILE), "scan": SCAN_NUMBER, "pixels_used": n_used,
    "calibration": {"gain_ev_per_channel": CALIB.gain, "offset_ev": CALIB.offset},
    "detector": {"material": DETECTOR.material, "noise_ev": DETECTOR.noise_ev,
                 "fano": DETECTOR.fano, "gamma_ev": DETECTOR.gamma_ev,
                 "sensor_thickness_um": DETECTOR.sensor_thickness_um,
                 "window": f"{DETECTOR.window_thickness_um} um {DETECTOR.window_material}",
                 "air_path_cm": DETECTOR.air_path_cm},
    "beamline": {"field_t": beamline.source.magnetic_field_t, "incident_air_m": AIR_PATH_M},
    "physics": {"level": LEVEL, "escape": USE_ESCAPE, "pileup_tau_us": TAU_US if USE_PILEUP else None,
                "secondary_fluorescence": bool(USE_MATRIX and SECONDARY_FLUORESCENCE)},
    "geometry": {"incidence_deg": GEOMETRY.incidence_deg, "takeoff_deg": GEOMETRY.takeoff_deg},
    "stack": [{"name": l.name, "formula": l.formula, "density": l.density_g_cm3,
               "thickness_um": l.thickness_um}
              for l in (QUANT["matrix"].stack if QUANT else STACK)],
    "target_layer": _s["target"],
    "elements": ELEMENTS, "families": list(FAMILIES), "e0_eff_kev": E0_EFF_KEV,
    "min_rel": MIN_REL, "roi_channels": list(ROI), "background_order": BKG_ORDER,
    "chi2_red": result.chi2_red,
    "residual_rms_percent_of_peak": float(100*np.sqrt((resid_rel**2).mean())),
    "parameters": {k: {"value": v, "stderr": e} for k, (v, e) in result.params().items()},
    "group_areas": {g.label: {"reference_line": g.reference.name,
                              "reference_area": result.areas[g.label],
                              "reference_area_stderr": result.area_errors[g.label],
                              "total_area": result.areas[g.label]*g.total_rel,
                              "ref_strength": g.ref_strength,
                              "n_lines": len(g.lines)}
                    for g in result.model.groups},
    "atom_ratios": None if QUANT is None else {
        "reference": _s["reference"], "ratios": QUANT["ratios"],
        "secondary_enhancement": QUANT["enhancement"], "absorption_factor": QUANT["absorption"],
        "fraction_from_other_layers": QUANT["other_layers"]},
}
(OUT_DIR / f"{stem}_fit.json").write_text(json.dumps(record, indent=2, default=float))
print("written:", *(p.name for p in sorted(OUT_DIR.glob(f"{stem}*"))), sep="\n  ")

## 10b. Detector resolution stability

The map fit holds the line widths fixed. If the detector resolution changes
during the scan (peaking time, temperature, a restart of the electronics),
pixels acquired before and after the change are fitted with the wrong widths,
which biases the areas of **overlapping** peaks (e.g. Ga Kb under the low-energy
side of As Ka) and shows up as a step in ratio maps and in the per-pixel chi2.
The per-pixel normalisation (monitor, live time) cannot do this: it scales every
area of a pixel by the same factor.

The diagnostic below follows the rms width and the centroid of a strong,
isolated peak row by row, in acquisition order, and flags abrupt steps:
a width step = resolution change, a centroid step = gain/offset change. Then,
depending on `RESOLUTION_MODE`:

* **segments**: the line widths are fitted on the average spectrum of each
  segment between steps, and each segment is solved with its own widths;
* **linear**: one extra column per pixel, the derivative of the modelled peaks
  with respect to a common scaling of all widths; its coefficient gives a map
  of the relative width change and absorbs small steps and drifts.

In [ ]:
n_slow, n_fast = scan.scan_shape
WIN = width_window(groups, result.areas, DETECTOR)
PROF = peak_width_profile(spectra, CALIB, WIN[:2], bin_size=n_fast)     # one bin per row
W_STEPS = detect_steps(PROF["width_ev"], window=5, threshold=STEP_THRESHOLD, min_step=STEP_MIN_EV)
C_STEPS = detect_steps(PROF["centroid_ev"], window=5, threshold=STEP_THRESHOLD, min_step=STEP_MIN_EV)

print(f"width diagnostic on {WIN[2]} ({WIN[0]/1e3:.2f}-{WIN[1]/1e3:.2f} keV), one value per row; "
      f"statistical error {np.nanmedian(PROF['width_err_ev']):.3f} eV per row")
slow_pos = scan.counters[scan.slow_motor_name].reshape(scan.scan_shape)[:, 0]
for r in W_STEPS:
    a, b = PROF["width_ev"][max(r-5, 0):r], PROF["width_ev"][r:r+5]
    print(f"WARNING: resolution step at row {r} ({scan.slow_motor_name} = {slow_pos[r]:.4f}): "
          f"rms width {np.mean(a):.2f} -> {np.mean(b):.2f} eV ({100*(np.mean(b)/np.mean(a)-1):+.1f} %)")
for r in C_STEPS:
    a, b = PROF["centroid_ev"][max(r-5, 0):r], PROF["centroid_ev"][r:r+5]
    print(f"WARNING: centroid (gain/offset) step at row {r}: {np.mean(a):.1f} -> {np.mean(b):.1f} eV "
          "-- the fixed calibration does not follow it")
if not (W_STEPS or C_STEPS):
    print("no resolution or calibration step detected")

# segments and per-segment line widths
USE_SEGMENTS = RESOLUTION_MODE in ("segments", "both") and len(W_STEPS) > 0
USE_WIDTH_TERM = RESOLUTION_MODE in ("linear", "both")
if USE_SEGMENTS:
    SEGMENTS = segment_labels(W_STEPS, n_slow, n_fast, n_pixels)
    SEG_FITS = fit_segments(corrected, spectra, scale, SEGMENTS, groups, CALIB, DETECTOR, ROI,
                            pileup=PILEUP if USE_PILEUP else None, **FIT_KW)
    MAP_SHAPE = {lab: r.popt for lab, r in SEG_FITS.items()}
    for lab, r in SEG_FITS.items():
        p = r.params()
        rows = np.unique(np.flatnonzero(SEGMENTS == lab) // n_fast)
        fw = DETECTOR.fwhm_ev(WIN[0] / 2 + WIN[1] / 2, noise_ev=p["noise_ev"][0], fano=p["fano"][0])
        print(f"segment {lab}: rows {rows[0]}-{rows[-1]}, FWHM({WIN[2]}) = {fw:.1f} eV, "
              f"gamma = {p['gamma_ev'][0]:.1f} eV, chi2_red = {r.chi2_red:.1f}")
else:
    SEGMENTS, SEG_FITS, MAP_SHAPE = None, None, result.popt
print(f"map fit: segments {'on' if USE_SEGMENTS else 'off'}, width term {'on' if USE_WIDTH_TERM else 'off'}")

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(7, 4.2), sharex=True, gridspec_kw={"hspace": 0.12})
rows = np.arange(PROF["width_ev"].size)
ax1.errorbar(rows, PROF["width_ev"], yerr=PROF["width_err_ev"], fmt="o-", ms=2.5, lw=0.8, color="0.2")
ax1.set_ylabel(f"{WIN[2]} rms width (eV)")
ax2.plot(rows, PROF["centroid_ev"], "o-", ms=2.5, lw=0.8, color="0.45")
ax2.set_ylabel("centroid (eV)"); ax2.set_xlabel("map row (slow axis, acquisition order)")
for r in W_STEPS:
    for ax in (ax1, ax2):
        ax.axvline(r - 0.5, color="C3", lw=1, ls="--")
ax1.set_title("Peak width and position along the scan" + (f" — steps at rows {W_STEPS}" if W_STEPS else ""),
              loc="left", fontsize=10)
fig.savefig(OUT_DIR / f"{H5_FILE.stem}_resolution_profile.png", dpi=150, bbox_inches="tight")

## 11. Per-pixel maps

With the peak positions and widths fixed by the average-spectrum fit, the model
is **linear** in the group amplitudes and the background coefficients. The whole
map is therefore one stacked linear solve: assemble `A^T W A` (only
`n_components x n_components` per pixel) and solve all pixels at once. The
per-pixel covariance comes out of the same factorisation, so every area map
arrives with an error map.

Weights are Poisson on the raw counts, `Var(y) = raw / scale^2`. Without them
the strongest channels dominate each pixel and the reported errors are
meaningless. The pile-up of each pixel (its own self-convolution times the
fitted resolving time) is subtracted before the solve.

In [ ]:
t0 = time.time()
maps = fit_map(spectra, scale, result.model, MAP_SHAPE, scan.scan_shape,
               pileup=PILEUP if USE_PILEUP else None,
               segments=SEGMENTS, width_correction=USE_WIDTH_TERM)
print(f"solved {spectra.shape[0]} pixels x {ROI[1]-ROI[0]} channels x "
      f"{len(maps.group_labels)+result.model.n_bkg} components in {time.time()-t0:.2f} s")
print(f"\nper-pixel reduced chi2: median {np.nanmedian(maps.chi2_red):.2f}  "
      f"(5-95 %: {np.nanpercentile(maps.chi2_red,5):.2f} - {np.nanpercentile(maps.chi2_red,95):.2f})")
print("A median near 1 is the real validation of the constrained model: it says the")
print("fixed line ratios and widths describe an INDIVIDUAL spectrum at the noise level.\n")
if maps.width_change is not None:
    wc = maps.width_change_map()
    print(f"relative width change from the linear term: median {100*np.nanmedian(wc):+.2f} %, "
          f"5-95 % range {100*np.nanpercentile(wc, 5):+.2f} to {100*np.nanpercentile(wc, 95):+.2f} %\n")
for lab in maps.group_labels:
    a, e = maps.area_map(lab, total=True), maps.error_map(lab, total=True)
    print(f"  {lab:<6} median {np.nanmedian(a):11.4g}   median S/N {np.nanmedian(a/e):7.1f}"
          f"   negative pixels {100*np.nanmean(a < 0):5.2f} %")

In [ ]:
# map of the residual width change (linear term), with the segment boundaries
if maps.width_change is not None:
    fig, ax = plt.subplots(figsize=(4.6, 3.8))
    im = ax.imshow(100 * maps.width_change_map(), origin="lower", cmap="RdBu_r",
                   vmin=-5, vmax=5, aspect="auto")
    if W_STEPS and USE_SEGMENTS:
        for r in W_STEPS:
            ax.axhline(r - 0.5, color="k", lw=0.8, ls="--")
    ax.set_xlabel(f"{scan.fast_motor_name} (pixel)"); ax.set_ylabel(f"{scan.slow_motor_name} (pixel)")
    ax.set_title("Width change left after the fixed shapes (%)", loc="left", fontsize=10)
    fig.colorbar(im, ax=ax, label="relative width change (%)")
    fig.savefig(OUT_DIR / f"{H5_FILE.stem}_width_change.png", dpi=150, bbox_inches="tight")

### Fit quality, group by group

A median per-pixel chi2 near 1 says the model is right *where the counts are*.
It says nothing about a group contributing a fraction of a per-cent of the
spectrum. `quality_report` separates the two: a group whose per-pixel S/N is of
order a few is detected but not quantifiable, and its map is noise.

In [ ]:
print(maps.quality_report())

### Individual pixel fits

Averaged residuals and a median chi2 both hide what a single spectrum looks
like. These panels are the direct check: whether the fixed line ratios hold
pixel by pixel, whether the background is plausible at the local count level,
and where the weak groups vanish into the noise.

Use `mode="worst"` when something looks wrong on a map — it shows the pixels
with the largest reduced chi2, which is where a missing element, a bad
background order or a pile-up artifact shows up first. `mode="spread"` walks the
chi2 distribution in equal quantiles for a representative rather than typical
sample.

In [ ]:
_pp = PILEUP if USE_PILEUP else None
fig, axes, pixels = plot_pixel_fits(spectra, scale, result.model, maps, MAP_SHAPE,
                                    n=10, mode="random", seed=0, ncols=5, pileup=_pp)
fig.savefig(OUT_DIR / f"{H5_FILE.stem}_pixel_fits_random.png", dpi=140, bbox_inches="tight")

# the diagnostic view: the ten worst-fitting pixels on the map
fig_w, _, pixels_w = plot_pixel_fits(spectra, scale, result.model, maps, MAP_SHAPE,
                                     n=10, mode="worst", ncols=5, pileup=_pp)
fig_w.savefig(OUT_DIR / f"{H5_FILE.stem}_pixel_fits_worst.png", dpi=140, bbox_inches="tight")
print("random pixels (slow, fast):", [(int(i)//maps.shape[1], int(i)%maps.shape[1]) for i in pixels])
print("worst  pixels (slow, fast):", [(int(i)//maps.shape[1], int(i)%maps.shape[1]) for i in pixels_w])

### Quantitative ratios

Each measured reference-line area is modelled as the sum over the layers of the
stack:

    A_g  ∝  w_g^t s_g^t + C_g,      s_g^k = (P^k + S^k) * eff,     C_g = sum_{k≠t} w_g^k s_g^k

`t` is the target layer, `w` mass fractions, `P` the primary fluorescence
integrated over the white beam (absorption in the layer and in all layers
above it, on the way in and out), `S` the secondary fluorescence (from exciting
lines in *any* layer, e.g. Ga/As K from the film exciting In L in the cap), and
`eff` the detection efficiency. Relative to the reference group `r`,

    w_i^t = [ (A_i / A_r) (w_r^t s_r^t + C_r) - C_i ] / s_i^t

so the known layers' contribution `C` is subtracted before the composition is
computed. Both `s` and `C` depend on the target composition, so
`quantify_layer` iterates it (cell 18). For a single-layer sample `C = 0` and
this is `n_i/n_r = (A_i/A_r)(s_r W_r)/(s_i W_i)`. `atom_ratio` applies the
converged sensitivities pixel by pixel (matrix taken as uniform over the map).

In [ ]:
G_Q = {g.label: g for g in groups}          # groups at the converged composition
REF = _s["reference"]
ref_el = G_Q[REF].element

if QUANT is not None:
    print(f"target layer, self-consistent composition (atoms per {ref_el} atom), "
          f"{QUANT['iterations']} iterations:")
    print(f"  {'group':<7}{'n/n_ref':>9}{'k':>9}{'SF/primary':>12}{'absorption':>12}{'other layers':>14}")
    for lab in [REF, *_s["analytes"]]:
        el = G_Q[lab].element
        print(f"  {lab:<7}{QUANT['ratios'][el]:>9.4f}{QUANT['k'].get(lab, 1.0):>9.4f}"
              f"{100*QUANT['enhancement'][lab]:>11.2f}%{QUANT['absorption'][lab]:>12.3f}"
              f"{100*QUANT['other_layers'][lab]:>13.1f}%")
    for el, n in _s["fixed_atoms"].items():
        print(f"  {el:<7}{n:>9.4f}  (fixed)")
    print("  'other layers' = share of the modelled intensity emitted by the non-target layers")
else:
    print("no matrix correction: ratios from the line-group sensitivities alone")
    for lab in _s["analytes"]:
        r, dr, k = number_ratio(result.areas[lab], result.areas[REF], G_Q[lab], G_Q[REF])
        print(f"  n_{G_Q[lab].element}/n_{ref_el} = {float(r):.4f}   (k = {k:.4f})")

# per-pixel atom-ratio maps
RATIO_MAPS = {}
for lab in _s["analytes"]:
    args = (maps.area_map(lab), maps.area_map(REF), maps.error_map(lab), maps.error_map(REF))
    if QUANT is not None:
        r, dr = atom_ratio(QUANT, lab, *args)
        k = QUANT["k"][lab]
    else:
        r, dr, k = number_ratio(args[0], args[1], G_Q[lab], G_Q[REF], args[2], args[3])
    RATIO_MAPS[lab] = (r, dr, k)

AN = _s["analytes"][0]                       # the main analyte, mapped below
x_map, dx_map, K_MAIN = RATIO_MAPS[AN]
v = x_map[np.isfinite(x_map)]
sig_stat = np.nanmedian(dx_map) * np.sqrt(np.nanmedian(maps.chi2_red))
an_el = G_Q[AN].element
print(f"\nmap of n_{an_el}/n_{ref_el}: median {np.median(v):.4f}")
print(f"  statistical error per pixel {sig_stat:.4f}; map spread {v.std():.4f}"
      f" -> excess (real) variation {np.sqrt(max(v.std()**2 - sig_stat**2, 0)):.4f}")

### Closure test and sensitivity of the soft lines

For a stoichiometric III-V layer the group-III atoms must add up to the group-V
ones: `n_Ga + n_In = n_As`. The test is sensitive mostly to In L, at 3.3 keV:

1. **Detection path.** With ~20 cm of air, only ~4 % of In La reaches the
   sensor, and every extra cm costs ~15 % more (against ~0.8 % for Ga Ka).
2. **Self-absorption.** In La has a ~3 µm attenuation length in (In,Ga)As, so
   in a 1.9 µm layer its intensity depends on the take-off angle and the
   thickness, unlike Ga K and As K.
3. **The InP cap.** 50 nm of InP emits ~6 % of the measured In L (excited by
   the beam and by Ga/As K fluorescence from the film) and absorbs ~2 % of the
   film's In La; both are modelled. Closing the balance with the cap alone
   would take ~500 nm of InP.
4. **Weak signal** on the worst-modelled part of the background.

Ga/As is robust against all of these: both are hard K lines a keV apart, and
the cap transmits >99.4 % of them. The table shows how the result moves with
the uncertain inputs.

In [ ]:
if _s.get("closure") and QUANT is not None:
    s_cl = sum(QUANT["ratios"].get(el, 0.0) for el in _s["closure"])
    print("closure: " + " + ".join(f"n_{el}" for el in _s["closure"]) + f" = {s_cl:.4f} "
          f"n_{ref_el}   (should be 1.000)\n")

    # Sensitivity: re-quantify (no refit) with one input changed at a time
    import dataclasses
    T = _s["target"]
    ST = QUANT["matrix"].stack

    def requant(detector=DETECTOR, geometry=GEOMETRY, stack=ST, target=T):
        mm = MatrixModel(stack, geometry, E_GRID_KEV, phi, secondary=SECONDARY_FLUORESCENCE,
                         target=target)
        return quantify_layer(result.areas, G_Q, mm, detector, REF, _s["analytes"],
                              _s["fixed_atoms"])["ratios"]

    cases = [("nominal", {})]
    for d in (-2.0, +2.0):
        cases.append((f"air {DETECTOR.air_path_cm + d:.0f} cm",
                      dict(detector=dataclasses.replace(DETECTOR, air_path_cm=DETECTOR.air_path_cm + d))))
    for d in (-10.0, +10.0):
        cases.append((f"take-off {GEOMETRY.takeoff_deg + d:.0f} deg",
                      dict(geometry=Geometry(GEOMETRY.incidence_deg, GEOMETRY.takeoff_deg + d))))
    if ST[T].thickness_um:
        for f in (0.8, 1.2):
            cases.append((f"{ST[T].name or 'target'} {f*ST[T].thickness_um:.2f} um",
                          dict(stack=ST.replace(T, ST[T].with_thickness(f * ST[T].thickness_um)))))
    for k in range(T):                                   # layers above the target
        lay = ST[k]
        for f in (0.6, 1.4):
            cases.append((f"{lay.name or k} {1000*f*lay.thickness_um:.0f} nm",
                          dict(stack=ST.replace(k, lay.with_thickness(f * lay.thickness_um)))))
        cases.append((f"no {lay.name or k}",
                      dict(stack=Stack([l for j, l in enumerate(ST) if j != k]), target=T - 1)))
    cases.append(("990 hPa", dict(detector=dataclasses.replace(DETECTOR, air_pressure_hpa=990.0))))

    els = [G_Q[l].element for l in _s["analytes"]]
    print(f"{'case':<20}" + "".join(f"{'n_'+e+'/n_'+ref_el:>13}" for e in els) + f"{'closure':>10}")
    for lab, kw in cases:
        rr = requant(**kw)
        print(f"{lab:<20}" + "".join(f"{rr[e]:>13.4f}" for e in els)
              + f"{sum(rr.get(e, 0) for e in _s['closure']):>10.4f}")
else:
    print("no closure relation defined for this sample")

In [ ]:
an_map, ref_map = maps.area_map(AN, total=True), maps.area_map(REF, total=True)
ratio, dratio = maps.ratio_map(AN, REF)
fast, slow = scan.fast_motor_name, scan.slow_motor_name
XX = (scan.counters[fast] - scan.counters[fast].min()).reshape(scan.scan_shape) * 1000.0
YY = (scan.counters[slow] - scan.counters[slow].min()).reshape(scan.scan_shape) * 1000.0

fig, axes = plt.subplots(2, 3, figsize=(15.5, 8.6))
def panel(ax, Z, title, cmap="viridis", cbl="", **kw):
    im = ax.pcolormesh(XX, YY, Z, cmap=cmap, shading="nearest", **kw)
    ax.set_aspect("equal"); ax.set_title(title, fontsize=11)
    ax.set_xlabel(f"{fast} (µm)"); ax.set_ylabel(f"{slow} (µm)")
    fig.colorbar(im, ax=ax, label=cbl, fraction=0.046)

lab_n = rf"$n_{{\rm {an_el}}}/n_{{\rm {ref_el}}}$"
panel(axes[0,0], an_map, f"{AN} total area", cbl="counts / live s (norm.)")
panel(axes[0,1], ref_map, f"{REF} total area", cbl="counts / live s (norm.)")
panel(axes[0,2], ratio, f"{AN} / {REF} intensity ratio", cmap="inferno", cbl="ratio")
lo, hi = np.nanpercentile(x_map, [1, 99])
panel(axes[1,0], x_map, lab_n, cmap="cividis", cbl=lab_n, vmin=lo, vmax=hi)
ax = axes[1,1]
ax.hist(v, bins=60, color="steelblue", edgecolor="black", lw=0.4, alpha=0.85)
ax.axvline(np.median(v), color="red", ls="--", lw=1.5, label=f"median {np.median(v):.4f}")
ax.set_xlabel(lab_n); ax.set_ylabel("pixels"); ax.legend(fontsize=9)
ax.set_title(f"spread {v.std():.4f}, statistical {sig_stat:.4f}", fontsize=10)
panel(axes[1,2], maps.chi2_red.reshape(scan.scan_shape), r"per-pixel reduced $\chi^2$",
      cmap="magma", cbl=r"$\chi^2_\nu$", vmin=0.9, vmax=1.4)
fig.suptitle(f"{SAMPLE} — per-pixel constrained Voigt fit, "
             f"{scan.scan_shape[0]}x{scan.scan_shape[1]} pixels", y=0.98)
fig.tight_layout()
fig.savefig(OUT_DIR / f"{H5_FILE.stem}_elemental_maps.png", dpi=150, bbox_inches="tight")

np.savez_compressed(OUT_DIR / f"{H5_FILE.stem}_maps.npz",
                    **{f"area_{l.replace(' ','')}": maps.area_map(l, total=True) for l in maps.group_labels},
                    **{f"err_{l.replace(' ','')}": maps.error_map(l, total=True) for l in maps.group_labels},
                    **{f"n_{G_Q[l].element}_over_n_{ref_el}": RATIO_MAPS[l][0] for l in RATIO_MAPS},
                    **{f"err_n_{G_Q[l].element}_over_n_{ref_el}": RATIO_MAPS[l][1] for l in RATIO_MAPS},
                    **{f"k_{G_Q[l].element}": np.array([RATIO_MAPS[l][2]]) for l in RATIO_MAPS},
                    chi2_red=maps.chi2_red.reshape(scan.scan_shape), x_um=XX, y_um=YY,
                    width_change=(maps.width_change_map() if maps.width_change is not None
                                  else np.full(scan.scan_shape, np.nan)),
                    segments=(maps.segments.reshape(scan.scan_shape) if maps.segments is not None
                              else np.zeros(scan.scan_shape, int)),
                    row_width_ev=PROF["width_ev"], row_centroid_ev=PROF["centroid_ev"],
                    resolution_steps=np.array(W_STEPS, dtype=int))
print("written:", (OUT_DIR / f"{H5_FILE.stem}_maps.npz").name,
      "and", (OUT_DIR / f"{H5_FILE.stem}_elemental_maps.png").name)

## Next steps

1. **Peak shape.** The residual is dominated by low-energy tailing on the
   strongest lines (with gamma fixed at a natural width, the low-energy sides of
   As Ka/Kb are underfit by tens of sigma). A Hypermet tail/shelf term addresses
   this; the free Lorentzian gamma imitates it with symmetric wings that
   overshoot the valleys between peaks.
2. **Geometry and detection path.** Confirm the incidence/take-off angles and
   the sample-detector air gap (or flush it with He): In L quantification is
   limited by them, not by statistics.
3. **Reference sample.** A GaAs wafer (exactly 1:1) in the same geometry would
   calibrate the Ga/As sensitivity and test the beam model; an InP or InAs
   reference would do the same for In L.
4. **Resolution tracking per pixel.** Steps are located at row granularity;
   a change in the middle of a row is caught by the linear width term only.
5. **Per-pixel weighting.** Two-pass Pearson weights in `fit_map` to remove the
   Neyman bias on weak groups.
6. **Energy calibration** as a fitted object with reported residuals, replacing
   the fixed gain/offset used here.